In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

# Live.watch — Folder Monitoring

Monitor a directory for new files. When Velox (or any tool) writes a new image, it auto-appears in the thumbnail panel. Simulated here with synthetic data written on a timer.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile, threading, time
import numpy as np
import torch
from quantem.widget import Live

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

def make_lattice(size=512, defocus=0.0, seed=42):
    rng = torch.Generator(device="cpu").manual_seed(seed)
    y = torch.linspace(-5, 5, size, device=device)
    x = torch.linspace(-5, 5, size, device=device)
    Y, X = torch.meshgrid(y, x, indexing="ij")
    a1 = torch.sin(2 * np.pi * (X + 0.5 * Y) * 2.5)
    a2 = torch.sin(2 * np.pi * Y * 2.5 * np.sqrt(3) / 2)
    lattice = (a1 + a2) ** 2
    sigma = 0.3 + abs(defocus) * 0.15
    blur = torch.exp(-(X**2 + Y**2) / (2 * sigma**2))
    image = lattice * (0.3 + 0.7 * blur)
    noise = torch.randn(size, size, generator=rng) * 0.05
    return (image.cpu() + noise).numpy().astype(np.float32)

## Start watching an empty folder

The widget shows "Waiting for data..." until the first file arrives.

In [ ]:
watch_dir = tempfile.mkdtemp(prefix="live_watch_")
w = Live.watch(watch_dir, pattern="*.npy", interval=1.0, title="Incoming Data")
w

## Simulate Velox writing files

Every 2 seconds a new "capture" arrives — mixed sizes (256, 512, 1024) to simulate overview + detail shots.

In [ ]:
def simulate_velox(folder, n=10, delay=2.0):
    sizes = [256, 512, 512, 1024, 512, 256, 512, 1024, 512, 512]
    for i in range(n):
        time.sleep(delay)
        sz = sizes[i % len(sizes)]
        img = make_lattice(sz, defocus=-2 + i * 0.4, seed=i)
        np.save(f"{folder}/capture_{i:03d}.npy", img)
        print(f"  Wrote capture_{i:03d}.npy ({sz}×{sz})")
    print("Acquisition complete.")
t = threading.Thread(target=simulate_velox, args=(watch_dir,), daemon=True)
t.start()
print("Simulating Velox writes... watch thumbnails appear above!")

## Check status

In [ ]:
w.summary()

## Stop watching

In [ ]:
w.stop()
print(f"Stopped. Total frames: {w.n_frames}")